<a href="https://colab.research.google.com/github/devwoo41/2026-OPT-1st/blob/wooju/BERT_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT on SQuAD: Pre-training vs Fine-tuning

이 노트북은 아래를 한 번에 수행합니다.

1. `bert-base-uncased`(사전학습만 된 모델 + QA head 랜덤 초기화) 성능 측정
2. SQuAD v1로 fine-tuning
3. fine-tuned 모델 성능 측정
4. 두 성능(EM, F1) 비교

> Colab 기준으로 작성되었습니다. GPU 런타임을 사용하세요.

In [1]:
# Colab 환경 패키지 설치 (버전 고정으로 호환성 확보)
!pip -q install -U "transformers==4.44.2" "datasets==2.21.0" "evaluate==0.4.2" "accelerate==0.34.2" sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
import os
import random
import inspect
import numpy as np
import pandas as pd
import torch
import transformers

from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    DefaultDataCollator,
    TrainingArguments,
    Trainer,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("transformers:", transformers.__version__)

device: cuda
transformers: 4.44.2


In [3]:
# 실험 설정 (속도-성능 균형 모드)
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 384
DOC_STRIDE = 128

# 너무 공격적으로 줄이지 않고 시간만 적당히 단축
MAX_TRAIN_SAMPLES = 10000
MAX_EVAL_SAMPLES = 1500

# A100/고성능 GPU에서는 16이 보통 안정적, CPU는 8 유지
BATCH_SIZE = 16 if torch.cuda.is_available() else 8
EPOCHS = 2
LEARNING_RATE = 2e-5

OUTPUT_DIR = "./bert_squad_ft"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
raw_datasets = load_dataset("squad")
raw_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [4]:
# 학습/평가 샘플 수 조절
train_examples = raw_datasets["train"]
val_examples = raw_datasets["validation"]

if MAX_TRAIN_SAMPLES is not None:
    train_examples = train_examples.select(range(min(MAX_TRAIN_SAMPLES, len(train_examples))))
if MAX_EVAL_SAMPLES is not None:
    val_examples = val_examples.select(range(min(MAX_EVAL_SAMPLES, len(val_examples))))

print(f"Train examples: {len(train_examples)}")
print(f"Validation examples: {len(val_examples)}")

def prepare_train_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    offset_mapping = inputs.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = inputs["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = inputs.sequence_ids(i)
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if offsets[token_start_index][0] > start_char or offsets[token_end_index][1] < end_char:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

def prepare_validation_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sequence_ids = inputs.sequence_ids(i)
        context_index = 1

        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        inputs["offset_mapping"][i] = [
            (o if sequence_ids[k] == context_index else None)
            for k, o in enumerate(inputs["offset_mapping"][i])
        ]

    inputs["example_id"] = example_ids
    return inputs

train_features = train_examples.map(
    prepare_train_features,
    batched=True,
    remove_columns=train_examples.column_names,
)

val_features = val_examples.map(
    prepare_validation_features,
    batched=True,
    remove_columns=val_examples.column_names,
)

print(train_features)
print(val_features)

Train examples: 10000
Validation examples: 1500


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 10138
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 1523
})


In [5]:
from collections import defaultdict

squad_metric = evaluate.load("squad")

def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = raw_predictions

    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = defaultdict(list)
    for i, f in enumerate(features):
        features_per_example[example_id_to_index[f["example_id"]]].append(i)

    predictions = {}

    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        context = example["context"]

        valid_answers = []
        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue
                    if end_index < start_index or (end_index - start_index + 1) > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    valid_answers.append(
                        {
                            "score": start_logits[start_index] + end_logits[end_index],
                            "text": context[start_char:end_char],
                        }
                    )

        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
            predictions[example["id"]] = best_answer["text"]
        else:
            predictions[example["id"]] = ""

    return predictions

def compute_squad_metrics(examples, features, raw_predictions):
    predictions = postprocess_qa_predictions(examples, features, raw_predictions)

    formatted_predictions = [{"id": k, "prediction_text": v} for k, v in predictions.items()]
    references = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]

    return squad_metric.compute(predictions=formatted_predictions, references=references)

In [6]:
# 1) Baseline: 사전학습만 된 BERT + 랜덤 QA head
baseline_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

baseline_args = TrainingArguments(
    output_dir="./baseline_eval_only",
    per_device_eval_batch_size=BATCH_SIZE,
    report_to="none",
)

baseline_trainer = Trainer(
    model=baseline_model,
    args=baseline_args,
    data_collator=DefaultDataCollator(),
)

baseline_raw_preds = baseline_trainer.predict(val_features)
baseline_metrics = compute_squad_metrics(val_examples, val_features, baseline_raw_preds.predictions)
print("Baseline metrics:", baseline_metrics)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Baseline metrics: {'exact_match': 0.3333333333333333, 'f1': 6.717789555588447}


In [7]:
# 2) Fine-tuning (안정 모드)
ft_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

# transformers 버전에 따라 eval_strategy / evaluation_strategy 중 하나를 사용
# 학습 중 평가는 완전히 끄고, 학습 후 Cell 9에서 최종 평가만 수행
# -> eval_loss 관련 KeyError를 구조적으로 방지

ta_params = inspect.signature(TrainingArguments.__init__).parameters
eval_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.06,
    save_strategy="no",
    load_best_model_at_end=False,
    dataloader_num_workers=2,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
training_kwargs[eval_key] = "no"

training_args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_features,
    data_collator=DefaultDataCollator(),
)

print("Training config loaded. Eval disabled during training; final metrics are computed in Cell 9.")
trainer.train()

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Training config loaded. Eval disabled during training; final metrics are computed in Cell 9.


Step,Training Loss
500,2.687000
1000,1.250500


TrainOutput(global_step=1268, training_loss=1.7804311276986395, metrics={'train_runtime': 75.8916, 'train_samples_per_second': 267.171, 'train_steps_per_second': 16.708, 'total_flos': 3973539779684352.0, 'train_loss': 1.7804311276986395, 'epoch': 2.0})

In [8]:
# 3) Fine-tuned 모델 평가
ft_raw_preds = trainer.predict(val_features)
ft_metrics = compute_squad_metrics(val_examples, val_features, ft_raw_preds.predictions)
print("Fine-tuned metrics:", ft_metrics)

Fine-tuned metrics: {'exact_match': 70.6, 'f1': 78.11128609993312}


In [9]:
# 4) 성능 비교 테이블
comparison_df = pd.DataFrame(
    [
        {
            "Model": "Pre-trained only (random QA head)",
            "Exact Match": baseline_metrics["exact_match"],
            "F1": baseline_metrics["f1"],
        },
        {
            "Model": "Fine-tuned on SQuAD",
            "Exact Match": ft_metrics["exact_match"],
            "F1": ft_metrics["f1"],
        },
    ]
)

comparison_df["EM Gain"] = comparison_df["Exact Match"] - comparison_df.loc[0, "Exact Match"]
comparison_df["F1 Gain"] = comparison_df["F1"] - comparison_df.loc[0, "F1"]

comparison_df

,Model,Exact Match,F1,EM Gain,F1 Gain
0,Pre-trained only (random QA head),0.333333,6.717790,0.000000,0.000000
1,Fine-tuned on SQuAD,70.600000,78.111286,70.266667,71.393497


## 실행 팁

- Colab에서 `런타임 > 런타임 유형 변경 > GPU` 선택 후 실행하세요.
- 오류가 난 뒤에는 반드시 `런타임 다시 시작` 후 1번 셀부터 순서대로 다시 실행하세요.
- 현재 기본값은 **속도-성능 균형 모드**입니다.
- 더 빠르게: `MAX_TRAIN_SAMPLES=6000`, `EPOCHS=1`.
- 더 정확하게: `MAX_TRAIN_SAMPLES=None`, `MAX_EVAL_SAMPLES=None`, `EPOCHS=2~3`.
- A100에서 배치가 안정적이면 `BATCH_SIZE=16` 유지, 메모리 이슈가 있으면 `8`로 낮추세요.